# Feline Skin Disease Detection - CNN Training\n\nRun all cells in order. Make sure you have a **GPU runtime** enabled:\n**Runtime → Change runtime type → GPU**

## 1. Verify GPU

In [ ]:
!pip install tensorflow-hub

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(" ", gpu)

TensorFlow version: 2.19.0
GPUs available: 0


## 2. Mount Google Drive & Clone Repo

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Ensure we're in a valid directory before cleanup
os.chdir('/content')
!rm -rf /content/repo

!git clone -b update/create-new-dataset-final https://github.com/pelta-ai/feline-skin-disease-detection.git /content/repo
%cd /content/repo

# Link Drive's final_data (train/val/test with images) into the repo
!rm -rf /content/repo/final_data
!ln -s /content/drive/MyDrive/feline-skin-disease-detection/final_data /content/repo/final_data

Mounted at /content/drive
Cloning into '/content/repo'...
remote: Enumerating objects: 5758, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 5758 (delta 23), reused 24 (delta 17), pack-reused 5721 (from 2)
Receiving objects: 100% (5758/5758), 249.66 MiB | 31.51 MiB/s, done.
Resolving deltas: 100% (607/607), done.
Updating files: 100% (4556/4556), done.
Error downloading object: trained_models/sample_cnn.keras (115814a): Smudge error: Error downloading trained_models/sample_cnn.keras (115814a54e4b8e9634d45a2aefb0e79dcbbc792e3360bfef53c51c5ed2fba0ac): batch response: This repository exceeded its LFS budget. The account responsible for the budget should increase it to restore access.

Errors logged to /content/repo/.git/lfs/logs/20260327T203151.77653656.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: trained_models/sample_cnn.keras: smudge filter lfs failed
You can

In [ ]:
import os

# Save trained models DIRECTLY to Google Drive so they survive runtime resets
DRIVE_MODELS = "/content/drive/MyDrive/feline-skin-disease-detection/trained_models"
os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f"Models will be saved to: {DRIVE_MODELS}")

## 3. Verify Data

In [ ]:
import os

for split in ["train", "val", "test"]:
    path = os.path.join("final_data", split)
    if os.path.exists(path):
        classes = sorted(os.listdir(path))
        total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
        print(f"{split}: {len(classes)} classes, {total} images - {classes}")
    else:
        print(f"WARNING: {path} not found!")

ConnectionAbortedError: [Errno 103] Software caused connection abort: 'new_data/train/demodicosis'

## 4. Train MobileNetV2 (5 seeds x 10 epochs)

In [ ]:
import os
import sys
import numpy as np

sys.path.insert(0, '/content/repo')

from src.classifiers import ClassifierFactory

seeds = [1, 2, 3, 4, 5]
v2_results = []

cnn_v2 = ClassifierFactory.create("mobilenet_v2")
cnn_v2.make_sub_datasets()

for s in seeds:
    save_name = f"new_mobilenetv2_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_v2.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_v2.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_v2.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    v2_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 5. MobileNetV2 Summary

In [ ]:
v2_accs = np.array([r["accuracy"] for r in v2_results])
v2_f1s = np.array([r["macro_f1"] for r in v2_results])

print("=== MobileNetV2 Summary ===")
print(f"Accuracy: {v2_accs.mean():.4f} +/- {v2_accs.std(ddof=1):.4f}")
print(f"Macro F1: {v2_f1s.mean():.4f} +/- {v2_f1s.std(ddof=1):.4f}")

## 6. Train MobileNetV3Small (5 seeds x 10 epochs)

In [ ]:
import importlib
import os
import sys
import src.classifiers.mobile_net_v3_small_classifier
import src.classifiers.classifier_factory
import src.classifiers
importlib.reload(src.classifiers.classifier_factory)
importlib.reload(src.classifiers)
from src.classifiers import ClassifierFactory

v3_results = []

cnn_v3 = ClassifierFactory.create("mobilenet_v3_small")
cnn_v3.make_sub_datasets()

for s in seeds:
    save_name = f"new_mobilenetv3small_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_v3.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_v3.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_v3.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    v3_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 7. MobileNetV3Small Summary

In [ ]:
v3_accs = np.array([r["accuracy"] for r in v3_results])
v3_f1s = np.array([r["macro_f1"] for r in v3_results])

print("=== MobileNetV3Small Summary ===")
print(f"Accuracy: {v3_accs.mean():.4f} +/- {v3_accs.std(ddof=1):.4f}")
print(f"Macro F1: {v3_f1s.mean():.4f} +/- {v3_f1s.std(ddof=1):.4f}")

## 8. Train ResNet50 (5 seeds x 10 epochs)

In [ ]:
import importlib
import os
import sys
import src.classifiers.resnet_50_classifier
import src.classifiers.classifier_factory
import src.classifiers
importlib.reload(src.classifiers.classifier_factory)
importlib.reload(src.classifiers)
from src.classifiers import ClassifierFactory

resnet50_results = []

cnn_resnet50 = ClassifierFactory.create("resnet50")
cnn_resnet50.make_sub_datasets()

for s in seeds:
    save_name = f"new_resnet50_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_resnet50.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_resnet50.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_resnet50.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    resnet50_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 9. ResNet50 Summary

In [ ]:
import numpy as np

resnet50_accs = np.array([r["accuracy"] for r in resnet50_results])
resnet50_f1s = np.array([r["macro_f1"] for r in resnet50_results])

print("=== ResNet50 Summary ===")
print(f"Accuracy: {resnet50_accs.mean():.4f} +/- {resnet50_accs.std(ddof=1):.4f}")
print(f"Macro F1: {resnet50_f1s.mean():.4f} +/- {resnet50_f1s.std(ddof=1):.4f}")

=== ResNet50 Summary ===
Accuracy: 0.9195 +/- 0.0098
Macro F1: 0.9102 +/- 0.0115


## 10. Train EfficientNetB0 (5 seeds x 10 epochs)

In [ ]:
import importlib
import os
import sys
import src.classifiers.efficient_net_b0_classifier
import src.classifiers.classifier_factory
import src.classifiers
importlib.reload(src.classifiers.classifier_factory)
importlib.reload(src.classifiers)
from src.classifiers import ClassifierFactory

effb0_results = []

cnn_effb0 = ClassifierFactory.create("efficientnet_b0")
cnn_effb0.make_sub_datasets()

for s in seeds:
    save_name = f"new_efficientnetb0_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_effb0.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_effb0.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_effb0.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    effb0_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 11. EfficientNetB0 Summary

In [ ]:
import numpy as np

effb0_accs = np.array([r["accuracy"] for r in effb0_results])
effb0_f1s = np.array([r["macro_f1"] for r in effb0_results])

print("=== EfficientNetB0 Summary ===")
print(f"Accuracy: {effb0_accs.mean():.4f} +/- {effb0_accs.std(ddof=1):.4f}")
print(f"Macro F1: {effb0_f1s.mean():.4f} +/- {effb0_f1s.std(ddof=1):.4f}")

=== EfficientNetB0 Summary ===
Accuracy: 0.9247 +/- 0.0036
Macro F1: 0.9169 +/- 0.0048


## 12. Train EfficientNetV2B0 (5 seeds x 10 epochs)

In [ ]:
import importlib
import os
import sys
import src.classifiers.efficient_net_v2_b0_classifier
import src.classifiers.classifier_factory
import src.classifiers
importlib.reload(src.classifiers.classifier_factory)
importlib.reload(src.classifiers)
from src.classifiers import ClassifierFactory

effv2b0_results = []

cnn_effv2b0 = ClassifierFactory.create("efficientnet_v2_b0")
cnn_effv2b0.make_sub_datasets()

for s in seeds:
    save_name = f"new_efficientnetv2b0_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_effv2b0.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_effv2b0.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_effv2b0.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    effv2b0_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 13. EfficientNetV2B0 Summary

In [ ]:
effv2b0_accs = np.array([r["accuracy"] for r in effv2b0_results])
effv2b0_f1s = np.array([r["macro_f1"] for r in effv2b0_results])

print("=== EfficientNetV2B0 Summary ===")
print(f"Accuracy: {effv2b0_accs.mean():.4f} +/- {effv2b0_accs.std(ddof=1):.4f}")
print(f"Macro F1: {effv2b0_f1s.mean():.4f} +/- {effv2b0_f1s.std(ddof=1):.4f}")

=== EfficientNetV2B0 Summary ===
Accuracy: 0.9052 +/- 0.0074
Macro F1: 0.8920 +/- 0.0087


## 14. Train NASNetMobile (5 seeds x 10 epochs)

In [ ]:
import importlib
import os
import sys
import src.classifiers.nasnet_mobile_classifier
import src.classifiers.classifier_factory
import src.classifiers
importlib.reload(src.classifiers.classifier_factory)
importlib.reload(src.classifiers)
from src.classifiers import ClassifierFactory

nasnet_results = []

cnn_nasnet = ClassifierFactory.create("nasnet_mobile")
cnn_nasnet.make_sub_datasets()

for s in seeds:
    save_name = f"new_nasnetmobile_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_nasnet.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_nasnet.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_nasnet.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    nasnet_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 15. NASNetMobile Summary

In [ ]:
nasnet_accs = np.array([r["accuracy"] for r in nasnet_results])
nasnet_f1s = np.array([r["macro_f1"] for r in nasnet_results])

print("=== NASNetMobile Summary ===")
print(f"Accuracy: {nasnet_accs.mean():.4f} +/- {nasnet_accs.std(ddof=1):.4f}")
print(f"Macro F1: {nasnet_f1s.mean():.4f} +/- {nasnet_f1s.std(ddof=1):.4f}")

=== NASNetMobile Summary ===
Accuracy: 0.9013 +/- 0.0148
Macro F1: 0.8897 +/- 0.0172


## 16. Train ConvNeXtTiny (5 seeds x 10 epochs)

In [ ]:
import importlib
import os
import sys
import src.classifiers.convnext_tiny_classifier
import src.classifiers.classifier_factory
import src.classifiers
importlib.reload(src.classifiers.classifier_factory)
importlib.reload(src.classifiers)
from src.classifiers import ClassifierFactory

convnext_results = []

cnn_convnext = ClassifierFactory.create("convnext_tiny")
cnn_convnext.make_sub_datasets()

for s in seeds:
    save_name = f"new_convnexttiny_frozen_seed_{s}.keras"
    save_full = os.path.join(DRIVE_MODELS, save_name)

    if os.path.exists(save_full):
        print(f"\n--- Skipping seed {s} (already on Drive), evaluating ---")
        result = cnn_convnext.evaluate(model_path=save_full)
    else:
        print(f"\n--- Training seed {s} ---")
        model = cnn_convnext.train_one_run(
            seed=s,
            save_name=save_name,
            save_path=DRIVE_MODELS,
            epochs=10,
            trainable_backbone=False,
            learning_rate=1e-3,
        )
        result = cnn_convnext.evaluate(model=model)

    result["seed"] = s
    result["save_name"] = save_name
    convnext_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 17. ConvNeXtTiny Summary

In [ ]:
convnext_accs = np.array([r["accuracy"] for r in convnext_results])
convnext_f1s = np.array([r["macro_f1"] for r in convnext_results])

print("=== ConvNeXtTiny Summary ===")
print(f"Accuracy: {convnext_accs.mean():.4f} +/- {convnext_accs.std(ddof=1):.4f}")
print(f"Macro F1: {convnext_f1s.mean():.4f} +/- {convnext_f1s.std(ddof=1):.4f}")

=== ConvNeXtTiny Summary ===
Accuracy: 0.9078 +/- 0.0116
Macro F1: 0.8992 +/- 0.0145


## 18. Comparison

In [ ]:
print("=== Comparison ===")
print(f"{'Model':<20} {'Accuracy':>18} {'Macro F1':>18}")
print("-" * 58)
print(f"{'MobileNetV2':<20} {v2_accs.mean():.4f} +/- {v2_accs.std(ddof=1):.4f}   {v2_f1s.mean():.4f} +/- {v2_f1s.std(ddof=1):.4f}")
print(f"{'MobileNetV3Small':<20} {v3_accs.mean():.4f} +/- {v3_accs.std(ddof=1):.4f}   {v3_f1s.mean():.4f} +/- {v3_f1s.std(ddof=1):.4f}")
print(f"{'EfficientNetB0':<20} {effb0_accs.mean():.4f} +/- {effb0_accs.std(ddof=1):.4f}   {effb0_f1s.mean():.4f} +/- {effb0_f1s.std(ddof=1):.4f}")
print(f"{'EfficientNetV2B0':<20} {effv2b0_accs.mean():.4f} +/- {effv2b0_accs.std(ddof=1):.4f}   {effv2b0_f1s.mean():.4f} +/- {effv2b0_f1s.std(ddof=1):.4f}")
print(f"{'NASNetMobile':<20} {nasnet_accs.mean():.4f} +/- {nasnet_accs.std(ddof=1):.4f}   {nasnet_f1s.mean():.4f} +/- {nasnet_f1s.std(ddof=1):.4f}")
print("-" * 58)
print(f"{'ResNet50':<20} {resnet50_accs.mean():.4f} +/- {resnet50_accs.std(ddof=1):.4f}   {resnet50_f1s.mean():.4f} +/- {resnet50_f1s.std(ddof=1):.4f}")
print(f"{'ConvNeXtTiny':<20} {convnext_accs.mean():.4f} +/- {convnext_accs.std(ddof=1):.4f}   {convnext_f1s.mean():.4f} +/- {convnext_f1s.std(ddof=1):.4f}")

=== Comparison ===
Model                          Accuracy           Macro F1
----------------------------------------------------------


NameError: name 'v2_accs' is not defined

## 34. Copy trained models to Google Drive

In [ ]:
import shutil

drive_models = "/content/drive/MyDrive/feline-skin-disease-detection/trained_models"
os.makedirs(drive_models, exist_ok=True)

for f in os.listdir("trained_models"):
    if f.endswith(".keras"):
        src = os.path.join("trained_models", f)
        dst = os.path.join(drive_models, f)
        shutil.copy2(src, dst)
        print(f"Copied {f} to Drive")

print("Done! Models saved to Google Drive.")

Copied resnet50_frozen_seed_3.keras to Drive
Copied efficientnetb0_frozen_seed_4.keras to Drive
Copied efficientnetb0_frozen_seed_5.keras to Drive
Copied convnexttiny_frozen_seed_4.keras to Drive
Copied convnexttiny_frozen_seed_2.keras to Drive
Copied nasnetmobile_frozen_seed_3.keras to Drive
Copied convnexttiny_frozen_seed_3.keras to Drive
Copied nasnetmobile_frozen_seed_2.keras to Drive
Copied resnet50_frozen_seed_4.keras to Drive
Copied resnet50_frozen_seed_5.keras to Drive
Copied efficientnetv2b0_frozen_seed_3.keras to Drive
Copied nasnetmobile_frozen_seed_5.keras to Drive
Copied resnet50_frozen_seed_2.keras to Drive
Copied efficientnetb0_frozen_seed_1.keras to Drive
Copied resnet50_frozen_seed_1.keras to Drive
Copied efficientnetv2b0_frozen_seed_2.keras to Drive
Copied efficientnetb0_frozen_seed_3.keras to Drive
Copied efficientnetv2b0_frozen_seed_4.keras to Drive
Copied efficientnetv2b0_frozen_seed_5.keras to Drive
Copied efficientnetv2b0_frozen_seed_1.keras to Drive
Copied nasne

In [ ]:
!rm -rf /content/repo/trained_models
!ln -s /content/drive/MyDrive/feline-skin-disease-detection/trained_models /content/repo/trained_models

In [ ]:
!ls /content/repo/trained_models/

sample_cnn_label_encoder.pkl  trained_models


In [ ]:
!ln -sf /content/drive/MyDrive/feline-skin-disease-detection/trained_models /content/repo/trained_models

In [ ]:
!python -m src.run_cnn

2026-03-27 20:36:59.548852: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774643819.574027    8839 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774643819.581651    8839 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774643819.602004    8839 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774643819.602097    8839 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774643819.602105    8839 computation_placer.cc:177] computation placer alr